# Retail Data Quality — Data Cleaning

## Objective

Clean the raw retail sales dataset while preserving data lineage and documenting every major cleaning decision.

The cleaning process will:

- Remove exact duplicate records
- Identify conflicting duplicate transaction IDs
- Standardize data types
- Standardize categorical values
- Handle missing values appropriately
- Validate numeric ranges
- Handle invalid dates
- Resolve or quarantine referential-integrity issues
- Recalculate derived financial fields
- Create a review dataset for records that cannot be safely corrected
- Produce a cleaning audit report

In [2]:
import pandas as pd
import numpy as np

from pathlib import Path

In [3]:
PROJECT_ROOT = Path.cwd().parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
CLEANED_DIR = PROJECT_ROOT / "data" / "cleaned"

CLEANED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [4]:
sales = pd.read_csv(
    RAW_DIR / "fact_sales.csv",
    low_memory=False
)

products = pd.read_csv(
    RAW_DIR / "dim_product.csv"
)

customers = pd.read_csv(
    RAW_DIR / "dim_customer.csv"
)

stores = pd.read_csv(
    RAW_DIR / "dim_store.csv"
)

print("Sales:", sales.shape)
print("Products:", products.shape)
print("Customers:", customers.shape)
print("Stores:", stores.shape)

Sales: (120680, 17)
Products: (500, 7)
Customers: (5000, 6)
Stores: (20, 5)


### Create a row-level lineage ID

In [5]:
sales["source_row_id"] = np.arange(
    1,
    len(sales) + 1
)

In [6]:
cleaned_sales = sales.copy()

print(
    "Starting rows:",
    len(cleaned_sales)
)

Starting rows: 120680


In [7]:
cleaning_audit = []

## Cleaning decision 
### 1 - Exact duplicates

In [8]:
exact_duplicate_mask = cleaned_sales.duplicated(
    keep="first"
)

exact_duplicate_count = exact_duplicate_mask.sum()

print(
    "Exact duplicate rows:",
    exact_duplicate_count
)

Exact duplicate rows: 0


In [9]:
cleaned_sales = cleaned_sales[
    ~exact_duplicate_mask
].copy()

In [10]:
cleaning_audit.append({
    "issue": "Exact duplicate rows",
    "action": "Removed duplicate copies",
    "records_affected": int(exact_duplicate_count),
    "business_reason": (
        "Exact duplicate sales can overstate transactions, "
        "units, revenue, and profit."
    )
})

### 2 - Conflicting transaction IDs

In [11]:
duplicate_id_mask = cleaned_sales[
    "transaction_id"
].duplicated(
    keep=False
)

duplicate_id_count = duplicate_id_mask.sum()

print(
    "Records with duplicate transaction IDs:",
    duplicate_id_count
)

Records with duplicate transaction IDs: 1358


In [12]:
duplicate_transaction_rows = (
    cleaned_sales[
        duplicate_id_mask
    ]
    .sort_values("transaction_id")
)

duplicate_transaction_rows.head(20)

,transaction_id,transaction_date,customer_id,product_id,store_id,quantity,unit_price,discount_amount,gross_sales,net_sales,unit_cost,cost_amount,profit_amount,gross_margin_pct,payment_method,sales_channel,category,source_row_id
115,T000116,2025-01-12 00:00:00.000000,C04209,P0241,S017,3,1603.57,962.14,4810.71,3848.57,1389.85,4169.55,-320.98,-8.34,Mada,Store,Beauty,116
120454,T000116,2025-01-12 00:00:00.000000,C04209,P0241,S017,3,1603.57,962.14,4810.71,3848.57,1389.85,4169.55,-320.98,-8.34,Mada,Store,Beauty,120455
201,T000202,2024-05-21 00:00:00.000000,C04453,P0052,S003,3,2884.67,0.00,8654.01,8654.01,1942.41,5827.23,2826.78,32.66,Card,E-commerce,Electronics,202
120013,T000202,2024-05-21 00:00:00.000000,C04453,P0052,S003,3,2884.67,0.00,8654.01,8654.01,1942.41,5827.23,2826.78,32.66,Card,E-commerce,Electronics,120014
327,T000328,2025-08-16 00:00:00.000000,C01576,P0134,S002,3,2806.55,841.96,8419.65,7577.69,1917.20,5751.60,1826.09,24.10,Mada,Store,Home Appliances,328
120572,T000328,2025-08-16 00:00:00.000000,C01576,P0134,S002,4,2806.55,841.96,11226.20,10384.24,1917.20,7668.80,2715.44,24.10,Mada,Store,Home Appliances,120573
120264,T000506,2024-01-24 00:00:00.000000,C00927,P0243,S013,2,1416.09,0.00,2832.18,2832.18,957.50,1915.00,917.18,32.38,STC Pay,Store,Electronics,120265
505,T000506,2024-01-24 00:00:00.000000,C00927,P0243,S013,2,1416.09,0.00,2832.18,2832.18,957.50,1915.00,917.18,32.38,STC Pay,Store,Electronics,506
543,T000544,2025-11-03 00:00:00.000000,C01525,P0006,S018,1,2071.30,414.26,2071.30,1657.04,1711.75,1711.75,-54.71,-3.30,STC Pay,E-commerce,Home Appliances,544
120073,T000544,2025-11-03 00:00:00.000000,C01525,P0006,S018,1,2071.30,414.26,2071.30,1657.04,1711.75,1711.75,-54.71,-3.30,STC Pay,E-commerce,Home Appliances,120074


In [13]:
duplicate_review = cleaned_sales[
    duplicate_id_mask
].copy()

In [14]:
cleaned_sales = cleaned_sales[
    ~duplicate_id_mask
].copy()

In [15]:
cleaning_audit.append({
    "issue": "Conflicting duplicate transaction IDs",
    "action": "Quarantined for review",
    "records_affected": int(duplicate_id_count),
    "business_reason": (
        "Conflicting records cannot be safely resolved "
        "without source-system evidence."
    )
})

### 3 - Clean the unit price data type

In [16]:
cleaned_sales["unit_price_raw"] = (
    cleaned_sales["unit_price"]
)

In [17]:
cleaned_sales["unit_price"] = (
    cleaned_sales["unit_price"]
    .astype("string")
    .str.replace(
        r"^\s*SAR\s*",
        "",
        regex=True
    )
    .str.replace(
        ",",
        "",
        regex=False
    )
)

In [18]:
cleaned_sales["unit_price"] = pd.to_numeric(
    cleaned_sales["unit_price"],
    errors="coerce"
)

In [19]:
cleaned_sales["unit_price"].dtype

Float64Dtype()

### Identify prices that still can not be converted

In [20]:
invalid_price_mask = (
    cleaned_sales["unit_price"].isna()
    |
    (cleaned_sales["unit_price"] <= 0)
)

invalid_price_count = invalid_price_mask.sum()

print(
    "Invalid unit prices:",
    invalid_price_count
)

Invalid unit prices: 150


In [21]:
invalid_price_review = cleaned_sales[
    invalid_price_mask
].copy()

In [22]:
cleaned_sales = cleaned_sales[
    ~invalid_price_mask
].copy()

In [23]:
cleaning_audit.append({
    "issue": "Invalid unit price",
    "action": "Quarantined for review",
    "records_affected": int(invalid_price_count),
    "business_reason": (
        "A non-positive or unresolvable selling price "
        "cannot support reliable revenue calculations."
    )
})

### 4  - Quantity

In [24]:
invalid_quantity_mask = (
    cleaned_sales["quantity"] <= 0
)

invalid_quantity_count = (
    invalid_quantity_mask.sum()
)

print(
    "Invalid quantities:",
    invalid_quantity_count
)

Invalid quantities: 492


In [25]:
invalid_quantity_review = cleaned_sales[
    invalid_quantity_mask
].copy()

In [26]:
cleaned_sales = cleaned_sales[
    ~invalid_quantity_mask
].copy()

In [27]:
cleaning_audit.append({
    "issue": "Zero or negative quantity",
    "action": "Quarantined for review",
    "records_affected": int(invalid_quantity_count),
    "business_reason": (
        "This fact table represents completed sales. "
        "Returns should be handled separately rather than "
        "silently interpreted as negative sales."
    )
})

### 5 - Missing transaction dates

In [28]:
cleaned_sales["transaction_date"] = pd.to_datetime(
    cleaned_sales["transaction_date"],
    errors="coerce"
)

In [29]:
missing_date_mask = (
    cleaned_sales["transaction_date"].isna()
)

missing_date_count = (
    missing_date_mask.sum()
)

print(
    "Missing/invalid dates:",
    missing_date_count
)

Missing/invalid dates: 149


In [30]:
missing_date_review = cleaned_sales[
    missing_date_mask
].copy()

In [31]:
cleaned_sales = cleaned_sales[
    ~missing_date_mask
].copy()

In [32]:
cleaning_audit.append({
    "issue": "Missing or invalid transaction date",
    "action": "Quarantined for review",
    "records_affected": int(missing_date_count),
    "business_reason": (
        "A transaction cannot be reliably assigned to a "
        "reporting period without a valid date."
    )
})

### 6 - Future dates

In [33]:
expected_min_date = pd.Timestamp("2024-01-01")
expected_max_date = pd.Timestamp("2025-12-31")

In [34]:
future_date_mask = (
    cleaned_sales["transaction_date"]
    > expected_max_date
)

future_date_count = future_date_mask.sum()

print(
    "Future-dated transactions:",
    future_date_count
)

Future-dated transactions: 99


In [35]:
future_date_review = cleaned_sales[
    future_date_mask
].copy()

In [36]:
cleaned_sales = cleaned_sales[
    ~future_date_mask
].copy()

In [37]:
cleaning_audit.append({
    "issue": "Future transaction date",
    "action": "Quarantined for review",
    "records_affected": int(future_date_count),
    "business_reason": (
        "Future dates fall outside the defined source-data "
        "reporting period and can distort time-based KPIs."
    )
})

### 7 - Missing payment methods

In [38]:
missing_payment_mask = (
    cleaned_sales["payment_method"].isna()
)

missing_payment_count = (
    missing_payment_mask.sum()
)

cleaned_sales["payment_method"] = (
    cleaned_sales["payment_method"]
    .fillna("Unknown")
)

In [39]:
cleaning_audit.append({
    "issue": "Missing payment method",
    "action": "Replaced with 'Unknown'",
    "records_affected": int(missing_payment_count),
    "business_reason": (
        "The transaction remains valid for sales reporting. "
        "Unknown preserves the sale without inventing a payment method."
    )
})

### 8 - Missing customer IDs

In [40]:
cleaned_sales["customer_id_raw"] = (
    cleaned_sales["customer_id"]
)

In [41]:
valid_customer_ids = set(
    customers["customer_id"]
)

missing_customer_mask = (
    cleaned_sales["customer_id"].isna()
)

orphan_customer_mask = (
    cleaned_sales["customer_id"].notna()
    &
    ~cleaned_sales["customer_id"].isin(
        valid_customer_ids
    )
)

In [42]:
print(
    "Missing customer IDs:",
    missing_customer_mask.sum()
)

print(
    "Orphan customer IDs:",
    orphan_customer_mask.sum()
)

Missing customer IDs: 594
Orphan customer IDs: 148


In [43]:
cleaned_sales.loc[
    missing_customer_mask | orphan_customer_mask,
    "customer_id"
] = "UNKNOWN"

In [44]:
customer_issue_count = (
    missing_customer_mask
    | orphan_customer_mask
).sum()

cleaning_audit.append({
    "issue": "Missing or orphan customer ID",
    "action": "Mapped to UNKNOWN",
    "records_affected": int(customer_issue_count),
    "business_reason": (
        "The sale remains valid, but customer attribution "
        "is unavailable. UNKNOWN preserves the transaction "
        "without inventing customer information."
    )
})

### 9 - Product and Store oprhan IDs

In [45]:
cleaned_sales["product_id_raw"] = (
    cleaned_sales["product_id"]
)

cleaned_sales["store_id_raw"] = (
    cleaned_sales["store_id"]
)

In [46]:
valid_product_ids = set(
    products["product_id"]
)

valid_store_ids = set(
    stores["store_id"]
)

orphan_product_mask = (
    ~cleaned_sales["product_id"]
    .isin(valid_product_ids)
)

orphan_store_mask = (
    ~cleaned_sales["store_id"]
    .isin(valid_store_ids)
)

print(
    "Orphan product IDs:",
    orphan_product_mask.sum()
)

print(
    "Orphan store IDs:",
    orphan_store_mask.sum()
)

Orphan product IDs: 148
Orphan store IDs: 99


In [47]:
cleaned_sales.loc[
    orphan_product_mask,
    "product_id"
] = "UNKNOWN"

cleaned_sales.loc[
    orphan_store_mask,
    "store_id"
] = "UNKNOWN"

In [48]:
cleaning_audit.append({
    "issue": "Orphan product ID",
    "action": "Mapped to UNKNOWN",
    "records_affected": int(orphan_product_mask.sum()),
    "business_reason": (
        "The sales event may still be valid, but product "
        "attribution cannot be trusted without a matching master record."
    )
})

cleaning_audit.append({
    "issue": "Orphan store ID",
    "action": "Mapped to UNKNOWN",
    "records_affected": int(orphan_store_mask.sum()),
    "business_reason": (
        "The sales event may still be valid, but store attribution "
        "cannot be trusted without a matching store master record."
    )
})

### 10 - Standardize categories

In [49]:
cleaning_audit.append({
    "issue": "Orphan product ID",
    "action": "Mapped to UNKNOWN",
    "records_affected": int(orphan_product_mask.sum()),
    "business_reason": (
        "The sales event may still be valid, but product "
        "attribution cannot be trusted without a matching master record."
    )
})

cleaning_audit.append({
    "issue": "Orphan store ID",
    "action": "Mapped to UNKNOWN",
    "records_affected": int(orphan_store_mask.sum()),
    "business_reason": (
        "The sales event may still be valid, but store attribution "
        "cannot be trusted without a matching store master record."
    )
})

In [50]:
category_standardization_map = {
    "electronics": "Electronics",
    "electronic": "Electronics",
    "home appliances": "Home Appliances",
    "fashion": "Fashion",
    "beauty": "Beauty",
    "sports": "Sports",
    "sport": "Sports",
    "grocery": "Grocery",
    "home & living": "Home & Living"
}

In [51]:
cleaned_sales["category"] = (
    cleaned_sales["category"]
    .str.lower()
    .map(category_standardization_map)
)

In [52]:
cleaned_sales["category"].value_counts(
    dropna=False
)

category
Electronics        19513
Home Appliances    19427
Grocery            16784
Home & Living      16539
Beauty             15784
Sports             15166
Fashion            15071
NaN                  148
Name: count, dtype: int64

### 11 - Invalid discounts

In [53]:
invalid_discount_mask = (
    (cleaned_sales["discount_amount"] < 0)
    |
    (
        cleaned_sales["discount_amount"]
        > cleaned_sales["gross_sales"]
    )
)

In [54]:
invalid_discount_count = (
    invalid_discount_mask.sum()
)

print(
    "Invalid discounts:",
    invalid_discount_count
)

Invalid discounts: 196


In [55]:
invalid_discount_review = cleaned_sales[
    invalid_discount_mask
].copy()

cleaned_sales = cleaned_sales[
    ~invalid_discount_mask
].copy()

In [56]:
cleaning_audit.append({
    "issue": "Invalid discount",
    "action": "Quarantined for review",
    "records_affected": int(invalid_discount_count),
    "business_reason": (
        "The intended discount cannot be reliably inferred. "
        "Capping the value would create an unsupported assumption."
    )
})

### 12 - Recalculate financial fields

In [57]:
cleaned_sales["gross_sales"] = (
    cleaned_sales["quantity"]
    * cleaned_sales["unit_price"]
).round(2)

In [58]:
cleaned_sales["net_sales"] = (
    cleaned_sales["gross_sales"]
    - cleaned_sales["discount_amount"]
).round(2)

In [59]:
cleaned_sales["cost_amount"] = (
    cleaned_sales["quantity"]
    * cleaned_sales["unit_cost"]
).round(2)

In [60]:
cleaned_sales["profit_amount"] = (
    cleaned_sales["net_sales"]
    - cleaned_sales["cost_amount"]
).round(2)

In [61]:
cleaned_sales["gross_margin_pct"] = np.where(
    cleaned_sales["net_sales"] != 0,
    (
        cleaned_sales["profit_amount"]
        / cleaned_sales["net_sales"]
        * 100
    ),
    np.nan
).round(2)

In [62]:
review_frames = [
    duplicate_review,
    invalid_price_review,
    invalid_quantity_review,
    missing_date_review,
    future_date_review,
    invalid_discount_review
]

review_sales = pd.concat(
    review_frames,
    ignore_index=True
).drop_duplicates(
    subset=["source_row_id"]
)

In [63]:
print(
    "Review records:",
    len(review_sales)
)

Review records: 2444


In [64]:
review_sales["review_reason"] = "Manual/source-system review required"

In [65]:
cleaned_sales.shape

(118236, 22)

In [66]:
cleaned_sales.head()

,transaction_id,transaction_date,customer_id,product_id,store_id,quantity,unit_price,discount_amount,gross_sales,net_sales,...,profit_amount,gross_margin_pct,payment_method,sales_channel,category,source_row_id,unit_price_raw,customer_id_raw,product_id_raw,store_id_raw
1,T000002,2025-04-28,C03663,P0169,S016,3,3819.65,0.00,11458.95,11458.95,...,2548.71,22.24,Card,Store,Home Appliances,2,3819.65,C03663,P0169,S016
2,T000003,2025-10-27,C04603,P0442,S017,6,2748.89,0.00,16493.34,16493.34,...,1910.52,11.58,Cash,Store,Fashion,3,2748.89,C04603,P0442,S017
3,T000004,2025-03-31,C04661,P0173,S013,7,3621.28,1267.45,25348.96,24081.51,...,4367.76,18.14,Card,E-commerce,Home & Living,4,3621.28,C04661,P0173,S013
4,T000005,2025-04-20,C02348,P0442,S009,2,2977.87,297.79,5955.74,5657.95,...,797.01,14.09,Mada,Store,Fashion,5,2977.87,C02348,P0442,S009
5,T000006,2024-07-28,C03271,P0447,S014,2,191.43,0.00,382.86,382.86,...,116.24,30.36,Card,Store,Home Appliances,6,191.43,C03271,P0447,S014


In [67]:
cleaned_sales.dtypes

transaction_id                 str
transaction_date    datetime64[us]
customer_id                    str
product_id                     str
store_id                       str
quantity                     int64
unit_price                 Float64
discount_amount            float64
gross_sales                Float64
net_sales                  Float64
unit_cost                  float64
cost_amount                float64
profit_amount              Float64
gross_margin_pct           float64
payment_method                 str
sales_channel                  str
category                       str
source_row_id                int64
unit_price_raw             float64
customer_id_raw                str
product_id_raw                 str
store_id_raw                   str
dtype: object

In [68]:
print(
    "Duplicate transaction IDs:",
    cleaned_sales["transaction_id"]
    .duplicated()
    .sum()
)

Duplicate transaction IDs: 0


In [69]:
print(
    "Exact duplicate rows:",
    cleaned_sales.duplicated().sum()
)

Exact duplicate rows: 0


In [70]:
print(
    "Invalid quantities:",
    (cleaned_sales["quantity"] <= 0).sum()
)

Invalid quantities: 0


In [71]:
print(
    "Invalid prices:",
    (
        cleaned_sales["unit_price"] <= 0
    ).sum()
)

Invalid prices: 0


In [72]:
print(
    "Missing dates:",
    cleaned_sales["transaction_date"].isna().sum()
)

print(
    "Future dates:",
    (
        cleaned_sales["transaction_date"]
        > expected_max_date
    ).sum()
)

Missing dates: 0
Future dates: 0


In [73]:
gross_check = (
    cleaned_sales["quantity"]
    * cleaned_sales["unit_price"]
).round(2)

print(
    "Gross sales mismatches:",
    ~np.isclose(
        cleaned_sales["gross_sales"],
        gross_check
    ).sum()
)

Gross sales mismatches: -118237


In [74]:
gross_mismatch = ~np.isclose(
    cleaned_sales["gross_sales"],
    gross_check,
    rtol=0,
    atol=0.01
)

print(
    "Gross sales mismatches:",
    gross_mismatch.sum()
)

Gross sales mismatches: 0


In [75]:
net_check = (
    cleaned_sales["gross_sales"]
    - cleaned_sales["discount_amount"]
).round(2)

net_mismatch = ~np.isclose(
    cleaned_sales["net_sales"],
    net_check,
    rtol=0,
    atol=0.01
)

print(
    "Net sales mismatches:",
    net_mismatch.sum()
)

Net sales mismatches: 0


In [76]:
profit_check = (
    cleaned_sales["net_sales"]
    - cleaned_sales["cost_amount"]
).round(2)

profit_mismatch = ~np.isclose(
    cleaned_sales["profit_amount"],
    profit_check,
    rtol=0,
    atol=0.01
)

print(
    "Profit mismatches:",
    profit_mismatch.sum()
)

Profit mismatches: 0


In [77]:
invalid_discounts_after = (
    (cleaned_sales["discount_amount"] < 0)
    |
    (
        cleaned_sales["discount_amount"]
        > cleaned_sales["gross_sales"]
    )
)

print(
    "Invalid discounts after cleaning:",
    invalid_discounts_after.sum()
)

Invalid discounts after cleaning: 0


In [78]:
cleaning_audit_df = pd.DataFrame(
    cleaning_audit
)

In [79]:
cleaning_audit_df

,issue,action,records_affected,business_reason
0,Exact duplicate rows,Removed duplicate copies,0,Exact duplicate sales can overstate transactio...
1,Conflicting duplicate transaction IDs,Quarantined for review,1358,Conflicting records cannot be safely resolved ...
2,Invalid unit price,Quarantined for review,150,A non-positive or unresolvable selling price c...
3,Zero or negative quantity,Quarantined for review,492,This fact table represents completed sales. Re...
4,Missing or invalid transaction date,Quarantined for review,149,A transaction cannot be reliably assigned to a...
5,Future transaction date,Quarantined for review,99,Future dates fall outside the defined source-d...
6,Missing payment method,Replaced with 'Unknown',358,The transaction remains valid for sales report...
7,Missing or orphan customer ID,Mapped to UNKNOWN,742,"The sale remains valid, but customer attributi..."
8,Orphan product ID,Mapped to UNKNOWN,148,"The sales event may still be valid, but produc..."
9,Orphan store ID,Mapped to UNKNOWN,99,"The sales event may still be valid, but store ..."


In [80]:
cleaned_sales_path = (
    CLEANED_DIR / "fact_sales_cleaned.csv"
)

cleaned_sales.to_csv(
    cleaned_sales_path,
    index=False
)

print(
    f"Saved cleaned data: {cleaned_sales_path}"
)

Saved cleaned data: d:\GitHub\retail_data_quality_project\data\cleaned\fact_sales_cleaned.csv


In [81]:
review_sales_path = (
    CLEANED_DIR / "fact_sales_review.csv"
)

review_sales.to_csv(
    review_sales_path,
    index=False
)

print(
    f"Saved review data: {review_sales_path}"
)

Saved review data: d:\GitHub\retail_data_quality_project\data\cleaned\fact_sales_review.csv


In [82]:
audit_path = (
    CLEANED_DIR / "cleaning_audit.csv"
)

cleaning_audit_df.to_csv(
    audit_path,
    index=False
)

print(
    f"Saved audit report: {audit_path}"
)

Saved audit report: d:\GitHub\retail_data_quality_project\data\cleaned\cleaning_audit.csv


In [83]:
print("=" * 50)
print("CLEANING SUMMARY")
print("=" * 50)

print(f"Raw records: {len(sales):,}")
print(f"Cleaned records: {len(cleaned_sales):,}")
print(f"Review records: {len(review_sales):,}")

print(
    f"Duplicate transaction IDs: "
    f"{cleaned_sales['transaction_id'].duplicated().sum():,}"
)

print(
    f"Invalid quantities: "
    f"{(cleaned_sales['quantity'] <= 0).sum():,}"
)

print(
    f"Invalid prices: "
    f"{(cleaned_sales['unit_price'] <= 0).sum():,}"
)

print(
    f"Invalid discounts: "
    f"{invalid_discounts_after.sum():,}"
)

print(
    f"Gross sales mismatches: "
    f"{gross_mismatch.sum():,}"
)

print(
    f"Profit mismatches: "
    f"{profit_mismatch.sum():,}"
)

CLEANING SUMMARY
Raw records: 120,680
Cleaned records: 118,236
Review records: 2,444
Duplicate transaction IDs: 0
Invalid quantities: 0
Invalid prices: 0
Invalid discounts: 0
Gross sales mismatches: 0
Profit mismatches: 0


In [84]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [85]:
fact_columns = [
    "transaction_id",
    "transaction_date",
    "customer_id",
    "product_id",
    "store_id",
    "quantity",
    "unit_price",
    "discount_amount",
    "gross_sales",
    "net_sales",
    "unit_cost",
    "cost_amount",
    "profit_amount",
    "gross_margin_pct",
    "payment_method",
    "sales_channel"
]

In [86]:
fact_sales_sql = cleaned_sales[
    fact_columns
].copy()

fact_sales_sql.to_csv(
    PROCESSED_DIR / "fact_sales_sql.csv",
    index=False
)

print(
    "SQL-ready fact table:",
    fact_sales_sql.shape
)

SQL-ready fact table: (118236, 16)


In [87]:
product_sql = products[
    [
        "product_id",
        "product_name",
        "category",
        "subcategory",
        "brand",
        "unit_cost",
        "selling_price"
    ]
].copy()

customer_sql = customers[
    [
        "customer_id",
        "customer_name",
        "gender",
        "age",
        "city",
        "customer_segment"
    ]
].copy()

store_sql = stores[
    [
        "store_id",
        "store_name",
        "city",
        "region",
        "store_type"
    ]
].copy()

In [88]:
product_sql.to_csv(
    PROCESSED_DIR / "dim_product_sql.csv",
    index=False
)

customer_sql.to_csv(
    PROCESSED_DIR / "dim_customer_sql.csv",
    index=False
)

store_sql.to_csv(
    PROCESSED_DIR / "dim_store_sql.csv",
    index=False
)